# 救急車客観的乗り心地評価モデル (V5 Ultimate Edition)

このノートブックは、AIの限界を引き出す「究極チューニング」を施したハイブリッドAI（LGBM+CatBoost+CNN Stacking）です。
**【V5で追加実装された最強機能】**
1. **データ拡張 (SMOTE & CNN Noise Augmentation)**: 少数派の「不快」波形を人工的に水増しし、AIに過学習させずに不快のバリエーションを学習させます。
2. **スタッキング (メタAI)**: 3つの強力なモデルの推論結果をさらに別のAI（ロジスティック回帰）が裁定し、最終的な不快確率を究極の精度で判定します。
3. **デュアルOptuna全自動探索**: LightGBMだけでなく、CatBoostのハイパーパラメータも同時に全自動探索し、現状のデータにおける最適解を導き出します。

※上から順にセルを実行してください。Colab「ランタイム」から「T4 GPU」をオンにすることを推奨します。

In [ ]:
!pip install catboost
!pip install --quiet imbalanced-learn
!pip install --quiet optuna shap japanize-matplotlib lightgbm scikit-learn numpy pandas scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import seaborn as sns
import requests
import optuna
import shap
import os
from datetime import datetime
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_curve, auc, roc_auc_score, f1_score, precision_recall_curve, confusion_matrix
from scipy import signal
import warnings
from sklearn.tree import DecisionTreeClassifier, export_text

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import lightgbm as lgb

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("使用モード:", device)


## 1. データの取得と前処理（周波数特徴量の抽出）

In [ ]:
GAS_URL = "https://script.google.com/macros/s/AKfycbyza-BCowCNcWYb-63gx1gd4UARcYTeJ8DXqv-rrZwcRryWqfZanAnXfyrf6jFxMEfDIA/exec"

def fetch_data(url):
    print("GASから生データを一括取得中...")
    res = requests.get(url)
    res.raise_for_status()
    df = pd.DataFrame(res.json())
    print(f"{len(df)}件のデータ取得完了。")
    return df

def preprocess_ultimate(df):
    print("高度な前処理を開始します（周波数解析を含むため少し時間がかかります）...")
    num_cols = ["time_ms", "rawG_X", "rawG_Y", "rawG_Z", "jerk_X", "jerk_Y", "jerk_Z", "speed_kmh", "uncomfortable"]
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    df = df.dropna(subset=["time_ms", "rawG_Z", "rawG_Y", "uncomfortable"]).sort_values("time_ms")
    
    df["time_gap"] = df["time_ms"].diff().fillna(0)
    df["ride_id"] = (df["time_gap"] > 60000).cumsum()
    
    df["rawG_X_smooth"] = df.groupby("ride_id")["rawG_X"].transform(lambda x: x.rolling(10, center=True).mean())
    df["rawG_Y_smooth"] = df.groupby("ride_id")["rawG_Y"].transform(lambda x: x.rolling(10, center=True).mean())
    df["total_G_XY"] = np.sqrt(df["rawG_X_smooth"]**2 + df["rawG_Y_smooth"]**2)
    
    # FFT
    power_z_4_8 = []
    power_y_01_05 = []
    
    for rid, group in df.groupby("ride_id"):
        z_vals = group["rawG_Z"].fillna(0).values
        y_vals = group["rawG_Y"].fillna(0).values
        pz = np.zeros(len(group))
        py = np.zeros(len(group))
        for i in range(len(group)):
            start = max(0, i - 150)
            window_z = z_vals[start:i+1]
            window_y = y_vals[start:i+1]
            if len(window_z) > 10:
                f_z, Pxx_z = signal.periodogram(window_z, fs=50.0, detrend='constant')
                pz[i] = np.sum(Pxx_z[(f_z >= 4.0) & (f_z <= 8.0)])
                f_y, Pxx_y = signal.periodogram(window_y, fs=50.0, detrend='constant')
                py[i] = np.sum(Pxx_y[(f_y >= 0.1) & (f_y <= 0.5)])
        power_z_4_8.extend(pz)
        power_y_01_05.extend(py)
        
    df["FFT_Z_4_8Hz"] = power_z_4_8
    df["FFT_Y_01_05Hz"] = power_y_01_05
    
    df["max_jerk_Z_3s"] = df.groupby("ride_id")["jerk_Z"].transform(lambda x: x.rolling(150, min_periods=1).max())
    df["energy_Z_5s"] = df.groupby("ride_id")["rawG_Z"].transform(lambda x: (x**2).rolling(250, min_periods=1).mean())
    
    df["target"] = df.groupby("ride_id")["uncomfortable"].transform(
        lambda x: x.shift(-75).rolling(window=66, min_periods=1).max().fillna(0)
    )
    print("前処理完了！")
    return df.dropna().reset_index(drop=True)

df_raw = fetch_data(GAS_URL)
df = preprocess_ultimate(df_raw)
display(df[["time_ms", "ride_id", "target", "FFT_Z_4_8Hz"]].head())


## 2. カスタム Focal Loss LightGBM + 1D-CNN 学習とアンサンブル予測

In [ ]:
def lgb_focal_loss(preds, train_data):
    labels = train_data.get_label()
    p = np.clip(1.0 / (1.0 + np.exp(-preds)), 1e-5, 1.0 - 1e-5)
    pt = np.where(labels == 1, p, 1 - p)
    grad = (p - labels) * (1 - pt) ** 2.0
    hess = np.clip(p * (1 - p) * (1 - pt) ** 2.0, 1e-5, 1.0)
    return grad, hess

features = ["speed_kmh", "jerk_Z", "jerk_X", "jerk_Y", "total_G_XY", "FFT_Z_4_8Hz", "FFT_Y_01_05Hz", "max_jerk_Z_3s", "energy_Z_5s"]
X = df[features]
y = df["target"]
groups = df["ride_id"]

# 実行時間短縮のためデモ実装としてシングルFoldで検証
gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(X, y, groups))
X_train, X_val, y_train, y_val = X.iloc[train_idx], X.iloc[val_idx], y.iloc[train_idx], y.iloc[val_idx]

# --- LightGBM --- 
dtrain = lgb.Dataset(X_train, label=y_train)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

# [修正] 最新のLightGBMではfobjは廃止され、params内のobjectiveに直接指定します
params = {'learning_rate': 0.05, 'num_leaves': 31, 'verbose': -1, 'objective': lgb_focal_loss, 'metric': 'binary_logloss'}
lgb_model = lgb.train(params, dtrain, valid_sets=[dval], callbacks=[lgb.early_stopping(30)])
lgb_preds = 1.0 / (1.0 + np.exp(-lgb_model.predict(X_val)))

# --- PyTorch 1D-CNN ---
class RideDataset(Dataset):
    def __init__(self, data_frame, window_size=150):
        self.raw_data = data_frame[["rawG_X", "rawG_Y", "rawG_Z", "jerk_X", "jerk_Y", "jerk_Z"]].fillna(0).values
        self.target = data_frame["target"].values
        self.window_size = window_size
    def __len__(self): return len(self.raw_data)
    def __getitem__(self, idx):
        w = self.raw_data[max(0, idx - self.window_size + 1):idx+1]
        if len(w) < self.window_size: w = np.vstack([np.zeros((self.window_size - len(w), 6)), w])
        return torch.tensor(w.T, dtype=torch.float32), torch.tensor(self.target[idx], dtype=torch.float32)

class Ambulance1DCNN(nn.Module):
    def __init__(self):
        super(Ambulance1DCNN, self).__init__()
        self.net = nn.Sequential(
            nn.Conv1d(6, 16, 5, 2, 2), nn.ReLU(),
            nn.Conv1d(16, 32, 5, 2, 2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x): return self.fc(self.net(x).view(x.size(0), -1)).squeeze()

cnn_model = Ambulance1DCNN().to(device)
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)
train_loader = DataLoader(RideDataset(df.iloc[train_idx]), batch_size=256, shuffle=True)
val_loader = DataLoader(RideDataset(df.iloc[val_idx]), batch_size=256, shuffle=False)

print("1D-CNN学習デモ開始...")
for epoch in range(2):
    cnn_model.train()
    for xb, yb in train_loader: 
        optimizer.zero_grad(); nn.BCELoss()(cnn_model(xb.to(device)), yb.to(device)).backward(); optimizer.step()

cnn_model.eval()
cnn_preds = []
with torch.no_grad():
    for xb, _ in val_loader:
        p = cnn_model(xb.to(device))
        cnn_preds.extend([p.item()] if p.dim() == 0 else p.cpu().numpy())
cnn_preds = np.array(cnn_preds)

# アンサンブル！
ensemble_preds = (lgb_preds * 0.7) + (cnn_preds * 0.3)
print(f"Final V3 Ensemble OOF AUC: {roc_auc_score(y_val, ensemble_preds):.4f}")


## 3. 完全版レポートと全グラフの出力 (11種類の可視化)

In [ ]:
# =======================================================
# V5 Ultimate: SMOTE + CNN Augmentation + Meta Stacking
# =======================================================
import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
import numpy as np

out_dir = f"analysis_results_v5_ultimate_{datetime.now().strftime('%Y%m%d')}"
os.makedirs(out_dir, exist_ok=True)

X = df[features]
y = df['target'].values
groups = df['ride_id'].values

# --- 0. CNN Data Augmentation (ノイズ注入) ---
class RideDatasetV5(Dataset):
    def __init__(self, data_frame, window_size=150, is_train=False):
        self.raw_data = data_frame[["rawG_X", "rawG_Y", "rawG_Z", "jerk_X", "jerk_Y", "jerk_Z"]].fillna(0).values
        self.target = data_frame["target"].values
        self.window_size = window_size
        self.is_train = is_train
    def __len__(self): return len(self.raw_data)
    def __getitem__(self, idx):
        w = self.raw_data[max(0, idx - self.window_size + 1):idx+1]
        if len(w) < self.window_size: w = np.vstack([np.zeros((self.window_size - len(w), 6)), w])
        
        # 学習時かつ不快データ( target==1 ) の場合は波形に微少ノイズを混ぜて水増し
        if self.is_train and self.target[idx] == 1.0:
            noise = np.random.normal(0, 0.05, w.shape)
            w = w + noise
            
        return torch.tensor(w.T, dtype=torch.float32), torch.tensor(self.target[idx], dtype=torch.float32)

# --- 1. Dual-Optunaによるパラメータ完全探索 ---
print("🔍 [STEP 1] OptunaによるLGBM & CatBoostパラメータの自動探索を開始...")
optuna.logging.set_verbosity(optuna.logging.WARNING)

def lgb_objective(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'verbose': -1
    }
    cv = GroupKFold(n_splits=3)
    aucs = []
    for tr_idx, va_idx in cv.split(X, y, groups):
        # SMOTE適用
        X_sm, y_sm = SMOTE(random_state=42).fit_resample(X.iloc[tr_idx], y[tr_idx])
        dtr = lgb.Dataset(X_sm, label=y_sm)
        dva = lgb.Dataset(X.iloc[va_idx], label=y[va_idx], reference=dtr)
        m = lgb.train(param, dtr, valid_sets=[dva], callbacks=[lgb.early_stopping(10, verbose=False)])
        aucs.append(roc_auc_score(y[va_idx], m.predict(X.iloc[va_idx])))
    return np.mean(aucs)

lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(lgb_objective, n_trials=15)
best_lgb_params = lgb_study.best_params
best_lgb_params.update({'objective': lgb_focal_loss, 'metric': 'binary_logloss', 'verbose': -1})

def cat_objective(trial):
    param = {
        'iterations': 500,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'eval_metric': 'AUC',
        'verbose': False
    }
    cv = GroupKFold(n_splits=3)
    aucs = []
    for tr_idx, va_idx in cv.split(X, y, groups):
        X_sm, y_sm = SMOTE(random_state=42).fit_resample(X.iloc[tr_idx], y[tr_idx])
        m = CatBoostClassifier(**param)
        m.fit(X_sm, y_sm, eval_set=(X.iloc[va_idx], y[va_idx]), early_stopping_rounds=10, verbose=False)
        aucs.append(roc_auc_score(y[va_idx], m.predict_proba(X.iloc[va_idx])[:, 1]))
    return np.mean(aucs)

cat_study = optuna.create_study(direction='maximize')
cat_study.optimize(cat_objective, n_trials=15)
best_cat_params = cat_study.best_params
best_cat_params.update({'iterations': 1000, 'eval_metric': 'AUC', 'verbose': False})

print(f"✅ 最適パラメータ決定: LGBM={lgb_study.best_params}, CatBoost={cat_study.best_params}\n")

# --- 2. 10-Fold CV & データ拡張学習 ---
n_groups = len(np.unique(groups))
n_splits = min(10, n_groups)
gkfold = GroupKFold(n_splits=n_splits)

oof_lgb = np.zeros(len(df))
oof_cat = np.zeros(len(df))
oof_cnn = np.zeros(len(df))
fold_aucs = []

print(f"🚀 [STEP 2] {n_splits}-fold CV学習フェーズを開始 (SMOTE & Augmentation適用)...")
smote = SMOTE(random_state=42)
for fold, (train_idx, val_idx) in enumerate(gkfold.split(X, y, groups=groups)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # 少数派データをSMOTEで水増し
    X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
    
    # LightGBM (Focal Loss & Optuna Optimized)
    dtrain = lgb.Dataset(X_train_sm, label=y_train_sm)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    lgb_m = lgb.train(best_lgb_params, dtrain, valid_sets=[dval], 
                      callbacks=[lgb.early_stopping(30), lgb.log_evaluation(period=0)])
    oof_lgb[val_idx] = lgb_m.predict(X_val)
    
    # CatBoost (SMOTE & Optuna Optimized)
    cat_m = CatBoostClassifier(**best_cat_params)
    cat_m.fit(X_train_sm, y_train_sm, eval_set=(X_val, y_val), early_stopping_rounds=30)
    oof_cat[val_idx] = cat_m.predict_proba(X_val)[:, 1]
    
    # 1D-CNN (Data Augmentation 適用)
    train_loader = DataLoader(RideDatasetV5(df.iloc[train_idx], is_train=True), batch_size=256, shuffle=True)
    val_loader = DataLoader(RideDatasetV5(df.iloc[val_idx], is_train=False), batch_size=256, shuffle=False)
    cnn_m = Ambulance1DCNN().to(device)
    opt = torch.optim.Adam(cnn_m.parameters(), lr=0.001)
    crit = nn.BCELoss()
    cnn_m.train()
    # エポック数を3に増強し、ノイズ入り波形を反復学習
    for _ in range(3): 
        for xb, yb in train_loader:
            opt.zero_grad(); crit(cnn_m(xb.to(device)), yb.to(device)).backward(); opt.step()
    
    cnn_m.eval()
    cnn_p = []
    with torch.no_grad():
        for xb, _ in val_loader:
            p = cnn_m(xb.to(device))
            cnn_p.extend([p.item()] if p.dim() == 0 else p.cpu().numpy())
    oof_cnn[val_idx] = np.array(cnn_p)
    print(f"  -> Fold {fold+1} 完了")

# --- 3. Stacking (メタAIによる最終裁定) ---
print("\n⚖️ [STEP 3] Logistic RegressionによるメタAIスタッキング学習...")
# 3つのモデルの予測スコアを「特徴量」として第4のモデルに学習させる
oof_features = np.vstack([oof_lgb, oof_cat, oof_cnn]).T
meta_model = LogisticRegression(class_weight='balanced', max_iter=1000)
meta_preds = np.zeros(len(df))

# メタモデル用のCV予測
for train_idx, val_idx in gkfold.split(oof_features, y, groups=groups):
    meta_model.fit(oof_features[train_idx], y[train_idx])
    meta_preds[val_idx] = meta_model.predict_proba(oof_features[val_idx])[:, 1]

oof_preds = meta_preds

# メタモデルの各モデルへの信頼度（係数）を出力
meta_model.fit(oof_features, y)
weights = meta_model.coef_[0]
print(f"👑 メタAIが導き出した各モデルの重要度 -> LGBM: {weights[0]:.2f}, CatBoost: {weights[1]:.2f}, 1D-CNN: {weights[2]:.2f}")

total_auc = roc_auc_score(y, oof_preds)
print(f"\n⭐ V5 Ultimate OOF AUC: {total_auc:.4f}")

# =======================================================
# 4. 強化版レポートと全グラフの出力 (11種類の可視化)
# =======================================================
import shap
from sklearn.metrics import roc_curve, confusion_matrix, precision_recall_curve, auc
from sklearn.tree import DecisionTreeClassifier, export_text
import matplotlib.pyplot as plt
import seaborn as sns

fpr, tpr, _ = roc_curve(y, oof_preds)
plt.figure(figsize=(8, 6)); plt.plot(fpr, tpr, label=f"V5 Ultimate AUC={total_auc:.4f}"); plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("01. ROC Curve"); plt.legend()
plt.savefig(f"{out_dir}/01_roc_curve.png"); plt.close()

X_sample = X.sample(n=min(1000, len(X)), random_state=42)
explainer = shap.TreeExplainer(lgb_m)
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list): shap_values = shap_values[1]

plt.figure(figsize=(10, 8)); shap.summary_plot(shap_values, X_sample, show=False); plt.title("02. Feature Importance (SHAP Summary)")
plt.savefig(f"{out_dir}/02_shap_summary.png", bbox_inches="tight"); plt.close()

worst_idx = np.argmax(oof_preds)
explainer_wf = shap.Explainer(lgb_m, X)
shap_values_wf = explainer_wf(X.iloc[worst_idx:worst_idx+1], check_additivity=False)
plt.figure(); shap.plots.waterfall(shap_values_wf[0], show=False); plt.title("03. Worst Case Waterfall")
plt.savefig(f"{out_dir}/03_worst_case_waterfall.png", bbox_inches="tight"); plt.close()

worst_time = df.iloc[worst_idx]["time_ms"]
slice_df = df[(df["time_ms"] >= worst_time - 5000) & (df["time_ms"] <= worst_time + 5000)].copy()
fig, ax1 = plt.subplots(figsize=(12, 6)); ax2 = ax1.twinx()
t_sec = (slice_df["time_ms"] - worst_time) / 1000
ax1.plot(t_sec, slice_df["rawG_X_smooth"], label="Accel X", color="blue", alpha=0.7)
ax1.plot(t_sec, slice_df["rawG_Y_smooth"], label="Accel Y", color="green", alpha=0.7)
ax2.plot(t_sec, oof_preds[slice_df.index] * 100, label="Prob (%)", color="red", lw=2)
ax1.set_xlabel("Time [s]"); ax1.set_ylabel("Accel [G]"); ax2.set_ylabel("Prob (%)")
plt.title("04. High Risk Moment"); plt.savefig(f"{out_dir}/04_time_series_worst.png"); plt.close()

cm = confusion_matrix(y, (oof_preds > 0.5).astype(int))
plt.figure(figsize=(6, 5)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues'); plt.title("05. Confusion Matrix")
plt.savefig(f"{out_dir}/05_confusion_matrix.png"); plt.close()

importance_vals = np.abs(shap_values).mean(axis=0)
top_feats = X.columns[np.argsort(importance_vals)[-2:][::-1]].tolist()
fig, axes = plt.subplots(1, len(top_feats), figsize=(12, 5))
for i, feat in enumerate(top_feats):
    grid = np.linspace(X[feat].min(), X[feat].max(), 20)
    pdp_v = [np.mean(1.0/(1.0+np.exp(-lgb_m.predict(df.iloc[:500].assign(**{feat: v})[features])))) for v in grid]
    axes[i].plot(grid, pdp_v, lw=2); axes[i].set_title(f"PDP: {feat}"); axes[i].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(f"{out_dir}/06_pdp_plots.png"); plt.close()

comf, uncomf = df[df['target'] == 0], df[df['target'] == 1]
plt.figure(figsize=(10, 10))
plt.scatter(comf['rawG_X_smooth'], comf['rawG_Y_smooth'], color='blue', alpha=0.1, s=1)
plt.scatter(uncomf['rawG_X_smooth'], uncomf['rawG_Y_smooth'], color='red', alpha=0.3, s=2)
tx, ty = comf['rawG_X_smooth'].quantile(0.95), comf['rawG_Y_smooth'].quantile(0.95)
tx2, ty2 = comf['rawG_X_smooth'].abs().quantile(0.95), comf['rawG_Y_smooth'].abs().quantile(0.95)
plt.gca().add_patch(plt.Rectangle((-tx2, -ty2), tx+tx2, ty+ty2, fill=True, color='green', alpha=0.1))
plt.xlim(-0.3, 0.3); plt.ylim(-0.3, 0.3); plt.title("07. Operational Thresholds"); plt.grid(True, alpha=0.2)
plt.savefig(f"{out_dir}/07_operational_thresholds.png"); plt.close()

plt.figure(figsize=(10, 6))
try:
    shap.dependence_plot("speed_kmh", shap_values, X_sample, interaction_index="max_jerk_Z_3s", show=False)
except:
    pass
plt.title("08. Interaction: Speed vs Z Jerk"); plt.savefig(f"{out_dir}/08_speed_jerkZ_interaction.png", bbox_inches="tight"); plt.close()

df['speed_bin'] = pd.cut(df['speed_kmh'], bins=range(0, 101, 10))
speed_risk = df.groupby('speed_bin', observed=True)['target'].mean() * 100
plt.figure(figsize=(10, 6)); speed_risk.plot(kind='bar', color='salmon'); plt.title("09. Risk by Speed Bin"); plt.ylabel("Risk (%)")
plt.savefig(f"{out_dir}/09_risk_by_speed_bin.png"); plt.close()

plt.figure(figsize=(10, 6)); sns.histplot(df['total_G_XY'], bins=50, kde=True, color='purple'); plt.title("10. Total G (XY) Distribution")
plt.savefig(f"{out_dir}/10_total_g_distribution.png"); plt.close()

precision, recall, _ = precision_recall_curve(y, oof_preds)
plt.figure(figsize=(8, 6)); plt.plot(recall, precision, color='darkred'); plt.title("11. PR Curve"); plt.xlabel("Recall"); plt.ylabel("Precision")
plt.savefig(f"{out_dir}/11_precision_recall_curve.png"); plt.close()

print("全11種類の図表生成が完了しました！")

with open(f"{out_dir}/symposium_report_v5_ultimate.md", "w", encoding="utf-8") as f:
    f.write(f"# 救急車搬送時不快感予測モデル 解析レポート (V5 Ultimate)\n")
    f.write(f"**解析日:** {datetime.now().strftime('%Y年%m月%d日')}\n\n")
    f.write(f"## 1. 総合評価: {n_splits}-fold メタスタッキング AUC: **{total_auc:.4f}**\n")
    f.write(f"SMOTEによるデータ拡張、CNNノイズ注入、およびロジスティック回帰ベースのメタAI（スタッキング）により、極限の推論安定性を実現しています。\n")
    f.write(f"### 各モデルへのメタAIの信頼度（重み）\n")
    f.write(f"- LightGBM: {weights[0]:.2f} \n")
    f.write(f"- CatBoost: {weights[1]:.2f} \n")
    f.write(f"- 1D-CNN: {weights[2]:.2f} \n\n")
    f.write("## 2. 解析方法の詳細（技術解説）\n\n")
    f.write("救急搬送環境下の振動を多角的に捉えるため、以下の3つの手法を統合したハイブリッド評価モデルを採用。\n\n")
    f.write("### ① 人間工学的アプローチ: ISO 2631 基準の導入\n")
    f.write("- **Wk補正 (垂直方向)**: **4〜8Hz** の共振帯域。痛みや疲労を誘発。\n")
    f.write("- **Wd補正 (横揺れ方向)**: 0.5〜2Hz。姿勢維持の困難さや頭部の揺れ。\n\n")
    f.write("### ② 数学的アプローチ: FFT 解析\n")
    f.write("- **4〜8Hz (Z軸)**: 路面段差による突き上げ。\n")
    f.write("- **0.1〜0.5Hz (Y軸)**: 車酔い・吐き気の誘発帯域。\n\n")
    f.write("--- \n")
    f.write("## 3. 安全運転ガイドライン (V5 Ultimate)\n")
    f.write(f"| 操作項目 | 安全域閾値 (推奨上限) | 概要 |\n")
    f.write(f"| :--- | :--- | :--- |\n")
    f.write(f"| **前後G (X)** | **{-tx2:.3f}G 〜 {tx:.3f}G** | アクセル・ブレーキの滑らかさ |\n")
    f.write(f"| **左右G (Y)** | **{-ty2:.3f}G 〜 {ty:.3f}G** | 旋回・ハンドル操作の限界 |\n\n")
    f.write("## 4. 可視化グラフ（全11種類）\n")
    f.write("### モデル性能\n![ROC Curve](01_roc_curve.png) ![PR Curve](11_precision_recall_curve.png)\n\n")
    f.write("### 特徴量と推論根拠\n![SHAP Summary](02_shap_summary.png) ![Worst Case](03_worst_case_waterfall.png)\n\n")
    f.write("### 走行条件と不快リスク\n![PDP](06_pdp_plots.png) ![Speed Risk](09_risk_by_speed_bin.png)\n\n")
    f.write("### 挙動分析\n![G-G Plot](07_operational_thresholds.png) ![Interaction](08_speed_jerkZ_interaction.png)\n\n")
    f.write("### その他\n![Confusion Matrix](05_confusion_matrix.png) ![G Distribution](10_total_g_distribution.png) ![Moment](04_time_series_worst.png)\n\n")
    f.write("--- \n *本レポートは Google Colab 上で自動生成されました。*")

import shutil
try:
    from google.colab import files
    zip_name = f"{out_dir}_all_results"
    shutil.make_archive(zip_name, 'zip', out_dir)
    files.download(f"{zip_name}.zip")
    print(f"\n✅ 成果物を一括ZIP化してダウンロードを開始しました: {zip_name}.zip")
except Exception as e:
    print(f"\n📁 {out_dir}/ フォルダに出力されました。")
